# 1. 已知固定位姿重建 coarse result

In [ ]:
import os
import time
from datetime import datetime

import numpy as np
import torch
import torch.nn.functional as F
import triton
import triton.language as tl

# ===============================================================
# CUDA 初始化
# ===============================================================
torch.cuda.init()
DEVICE_ID = 1
torch.cuda.set_device(DEVICE_ID)
device = torch.device(f"cuda:{DEVICE_ID}")
print(f"[INIT] 当前运行设备: {device}")

# ===============================================================
# 声学参数 / 分辨率
# ===============================================================
sigma = 0.1e-3     # m，既作为核参数也作为网格步长（分辨率）
vs = 1500.0        # m/s
delta_time = 120e-9 # s, 采样间隔

# ===============================================================
# 仿真时间范围：起始下标 / 结束下标（不包含结束下标）
# ===============================================================
t_start_idx = 225  # 包含.   如不需要，改为 0
t_end_idx   = 415  # 不包含   如不需要，改为 采样点数
# ===============================================================
# 探头位置 & 参考信号（仅用于尺寸和 target）
# ===============================================================
loc_all = np.loadtxt("data/kidney_loc_sym_A_512.txt")
loc_all = loc_all*1.0e-3
signals_all = np.loadtxt("data/kidney_data_sym_A_512.txt") / 100
n_sensors_full, n_time = signals_all.shape
n_time_sub = t_end_idx - t_start_idx
print(f"[INFO] sensors={n_sensors_full}, full time={n_time}, sim time range=({t_start_idx}, {t_end_idx}), len={n_time_sub}")

# （可选）下采样探头以降低计算量
sel_step = 1  # 每隔 sel_step 取一个探头，1 表示全取
sel_idx = np.arange(0, loc_all.shape[0], sel_step)
loc = loc_all[sel_idx]  # (N_sensors, 3)
signals = signals_all[sel_idx][:, t_start_idx:t_end_idx]  # 只取时间段
n_sensors = loc.shape[0]
print(f"[INFO] 使用探头数={n_sensors} (每 {sel_step} 个取一个)")

sens_x = torch.tensor(loc[:, 0], dtype=torch.float32, device=device).contiguous()
sens_y = torch.tensor(loc[:, 1], dtype=torch.float32, device=device).contiguous()
sens_z = torch.tensor(loc[:, 2], dtype=torch.float32, device=device).contiguous()

# ===============================================================
# 源点：基于分辨率 sigma 的 256^3 网格初始化
# ===============================================================
GRID_SIZE = 256
voxel_size = sigma  # 网格步长 = 分辨率
center = (GRID_SIZE - 1) / 2.0
coords = (np.arange(GRID_SIZE) - center) * voxel_size  # 从 -center*voxel_size 到 +center*voxel_size
xg, yg, zg = np.meshgrid(coords, coords, coords, indexing="ij")
src_x_np = xg.astype(np.float32).ravel()
src_y_np = yg.astype(np.float32).ravel()
src_z_np = zg.astype(np.float32).ravel()
n_sources = src_x_np.size
print(f"[INFO] grid sources = {n_sources} ({GRID_SIZE}^3), step={voxel_size} m")

src_x = torch.tensor(src_x_np, device=device).contiguous()
src_y = torch.tensor(src_y_np, device=device).contiguous()
src_z = torch.tensor(src_z_np, device=device).contiguous()

# 源强度 Pc 作为可训练参数
Pc_param = torch.nn.Parameter(torch.randn(n_sources, device=device))

# ===============================================================
# Triton 前向核：Gaussian
# out[s, t] = sum_k Pc[k] * 0.5 * ((r_k - vs*t)/r_k) * exp(-(r_k - vs*t)^2 / (2*sigma^2))
# ===============================================================
@triton.jit
def forward_kernel(
    Pc_ptr, src_x_ptr, src_y_ptr, src_z_ptr,
    sens_x_ptr, sens_y_ptr, sens_z_ptr,
    out_ptr,
    n_sources, n_time_sub,
    t_start_idx,
    delta_t, vs, sigma,
    stride_out_s, stride_out_t,
    BLOCK_K: tl.constexpr, BLOCK_T: tl.constexpr
):
    pid_t = tl.program_id(0)  # 时间块
    pid_s = tl.program_id(1)  # 探头
    t_offsets = tl.arange(0, BLOCK_T)
    t_idx_rel = pid_t * BLOCK_T + t_offsets
    t_mask = t_idx_rel < n_time_sub
    t_vals = (t_start_idx + t_idx_rel) * delta_t  # [T]

    sx = tl.load(sens_x_ptr + pid_s)
    sy = tl.load(sens_y_ptr + pid_s)
    sz = tl.load(sens_z_ptr + pid_s)

    acc = tl.zeros((BLOCK_T,), dtype=tl.float32)

    for k in range(0, n_sources, BLOCK_K):
        idx_k = k + tl.arange(0, BLOCK_K)
        mask_k = idx_k < n_sources

        px = tl.load(src_x_ptr + idx_k, mask=mask_k, other=0.0)
        py = tl.load(src_y_ptr + idx_k, mask=mask_k, other=0.0)
        pz = tl.load(src_z_ptr + idx_k, mask=mask_k, other=0.0)
        pc = tl.load(Pc_ptr    + idx_k, mask=mask_k, other=0.0)

        dx = sx - px
        dy = sy - py
        dz = sz - pz
        r = tl.sqrt(dx * dx + dy * dy + dz * dz + 1e-12)  # [K]

        r_mat  = r[:, None]      # [K,1]
        pc_mat = pc[:, None]     # [K,1]
        t_mat  = t_vals[None, :] # [1,T]

        rt = r_mat - vs * t_mat
        contrib = pc_mat * 0.5 * (rt / r_mat) * tl.exp(-(rt * rt) / (2.0 * sigma * sigma))
        contrib = tl.where(mask_k[:, None], contrib, 0.0)
        acc += tl.sum(contrib, axis=0)

    out_ptrs = out_ptr + pid_s * stride_out_s + t_idx_rel * stride_out_t
    tl.store(out_ptrs, acc, mask=t_mask)

# ===============================================================
# Triton 反向核：对 Pc 求梯度
# ===============================================================
@triton.jit
def backward_kernel(
    src_x_ptr, src_y_ptr, src_z_ptr,
    sens_x_ptr, sens_y_ptr, sens_z_ptr,
    grad_out_ptr, grad_pc_ptr,
    n_sources, n_sensors, n_time_sub,
    t_start_idx,
    delta_t, vs, sigma,
    stride_go_s, stride_go_t,
    BLOCK_K: tl.constexpr, BLOCK_T: tl.constexpr
):
    pid_t = tl.program_id(0)
    pid_s = tl.program_id(1)
    t_offsets = tl.arange(0, BLOCK_T)
    t_idx_rel = pid_t * BLOCK_T + t_offsets
    t_mask = t_idx_rel < n_time_sub
    t_vals = (t_start_idx + t_idx_rel) * delta_t

    sx = tl.load(sens_x_ptr + pid_s)
    sy = tl.load(sens_y_ptr + pid_s)
    sz = tl.load(sens_z_ptr + pid_s)

    go_ptrs = grad_out_ptr + pid_s * stride_go_s + t_idx_rel * stride_go_t
    go = tl.load(go_ptrs, mask=t_mask, other=0.0)  # [T]

    for k in range(0, n_sources, BLOCK_K):
        idx_k = k + tl.arange(0, BLOCK_K)
        mask_k = idx_k < n_sources

        px = tl.load(src_x_ptr + idx_k, mask=mask_k, other=0.0)
        py = tl.load(src_y_ptr + idx_k, mask=mask_k, other=0.0)
        pz = tl.load(src_z_ptr + idx_k, mask=mask_k, other=0.0)

        dx = sx - px
        dy = sy - py
        dz = sz - pz
        r = tl.sqrt(dx * dx + dy * dy + dz * dz + 1e-12)  # [K]

        r_mat = r[:, None]
        t_mat = t_vals[None, :]

        rt = r_mat - vs * t_mat
        dfdpc = 0.5 * (rt / r_mat) * tl.exp(-(rt * rt) / (2.0 * sigma * sigma))
        dfdpc = tl.where(mask_k[:, None], dfdpc, 0.0)

        grad_k = tl.sum(dfdpc * go[None, :], axis=1)  # [K]
        tl.atomic_add(grad_pc_ptr + idx_k, grad_k, mask=mask_k)

# ===============================================================
# 自定义 Autograd Function
# ===============================================================
class GaussianSimFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx,
                Pc, src_x, src_y, src_z,
                sens_x, sens_y, sens_z,
                t_start_idx, n_time_sub,
                delta_t, vs, sigma):
        out = torch.empty((sens_x.numel(), n_time_sub), device=Pc.device, dtype=torch.float32)

        BLOCK_T = 128
        BLOCK_K = 128
        grid = (triton.cdiv(n_time_sub, BLOCK_T), sens_x.numel())

        forward_kernel[grid](
            Pc, src_x, src_y, src_z,
            sens_x, sens_y, sens_z,
            out,
            Pc.numel(), n_time_sub,
            t_start_idx,
            delta_t, vs, sigma,
            out.stride(0), out.stride(1),
            BLOCK_K=BLOCK_K, BLOCK_T=BLOCK_T,
            num_warps=4, num_stages=2
        )
        ctx.save_for_backward(src_x, src_y, src_z, sens_x, sens_y, sens_z)
        ctx.n_sources = Pc.numel()
        ctx.n_sensors = sens_x.numel()
        ctx.n_time_sub = n_time_sub
        ctx.t_start_idx = t_start_idx
        ctx.delta_t = delta_t
        ctx.vs = vs
        ctx.sigma = sigma
        return out

    @staticmethod
    def backward(ctx, grad_out):
        src_x, src_y, src_z, sens_x, sens_y, sens_z = ctx.saved_tensors
        grad_pc = torch.zeros(ctx.n_sources, device=grad_out.device, dtype=torch.float32)

        BLOCK_T = 128
        BLOCK_K = 128
        grid = (triton.cdiv(ctx.n_time_sub, BLOCK_T), ctx.n_sensors)

        backward_kernel[grid](
            src_x, src_y, src_z,
            sens_x, sens_y, sens_z,
            grad_out.contiguous(), grad_pc,
            ctx.n_sources, ctx.n_sensors, ctx.n_time_sub,
            ctx.t_start_idx,
            ctx.delta_t, ctx.vs, ctx.sigma,
            grad_out.stride(0), grad_out.stride(1),
            BLOCK_K=BLOCK_K, BLOCK_T=BLOCK_T,
            num_warps=4, num_stages=2
        )
        return grad_pc, None, None, None, None, None, None, None, None, None, None, None

def gaussian_sim(Pc, src_x, src_y, src_z, sens_x, sens_y, sens_z,
                 t_start_idx, n_time_sub, delta_t, vs, sigma):
    return GaussianSimFunction.apply(Pc, src_x, src_y, src_z,
                                     sens_x, sens_y, sens_z,
                                     t_start_idx, n_time_sub,
                                     delta_t, vs, sigma)

# ===============================================================
# TGV 正则
# ===============================================================
def forward_diff(tensor, dim):
    diff = torch.roll(tensor, shifts=-1, dims=dim) - tensor
    idx = [slice(None)] * tensor.ndim
    idx[dim] = slice(-1, None)
    diff[tuple(idx)] = 0.0
    return diff

def tgv2_regularization(A_flat, grid_shape, alpha0=2.0, alpha1=1.0, eps=1e-8):
    A = A_flat.view(*grid_shape)
    gx = forward_diff(A, 0)
    gy = forward_diff(A, 1)
    gz = forward_diff(A, 2)
    grad_mag = torch.sqrt(gx * gx + gy * gy + gz * gz + eps)
    first_term = grad_mag.mean()

    gxx = forward_diff(gx, 0)
    gyy = forward_diff(gy, 1)
    gzz = forward_diff(gz, 2)
    gxy = 0.5 * (forward_diff(gx, 1) + forward_diff(gy, 0))
    gxz = 0.5 * (forward_diff(gx, 2) + forward_diff(gz, 0))
    gyz = 0.5 * (forward_diff(gy, 2) + forward_diff(gz, 1))
    second_sq = gxx * gxx + gyy * gyy + gzz * gzz \
                + 2.0 * (gxy * gxy + gxz * gxz + gyz * gyz)
    second_term = torch.sqrt(second_sq + eps).mean()
    return alpha1 * first_term + alpha0 * second_term

GRID_SHAPE = (GRID_SIZE, GRID_SIZE, GRID_SIZE)
lambda_tv = 5.0
tgv_alpha0 = 2.0
tgv_alpha1 = 1.0

# ===============================================================
# 训练循环
# ===============================================================
if __name__ == "__main__":
    print("\n[Train] 开始训练循环 (直角坐标 Gaussian)...")
    target = torch.tensor(signals, dtype=torch.float32, device=device).contiguous()

    optimizer = torch.optim.Adam([Pc_param], lr=0.5)
    max_epochs = 200
    root_ckpt_dir = "checkpoints_12"
    os.makedirs(root_ckpt_dir, exist_ok=True)

    for epoch in range(max_epochs):
        t_start = time.time()

        optimizer.zero_grad()
        Pc = F.softplus(Pc_param)  # 保持非负

        y_pred = gaussian_sim(Pc, src_x, src_y, src_z, sens_x, sens_y, sens_z,
                              t_start_idx, n_time_sub, delta_time, vs, sigma)

        data_loss = torch.mean((y_pred - target) ** 2)
        tv_loss = tgv2_regularization(Pc, GRID_SHAPE, alpha0=tgv_alpha0, alpha1=tgv_alpha1)
        loss = data_loss + lambda_tv * tv_loss

        loss.backward()
        optimizer.step()

        epoch_time = time.time() - t_start
        grad_mean = Pc_param.grad.mean().item() if Pc_param.grad is not None else 0.0
        print(f"[Epoch {epoch+1}/{max_epochs}] "
              f"loss={loss.item():.6e}, data={data_loss.item():.6e}, tv={tv_loss.item():.6e}, "
              f"grad_mean={grad_mean:.4e}, time={epoch_time:.2f}s, "
              f"Pc_min={Pc.min().item():.3e}, Pc_mean={Pc.mean().item():.3e}")

        # 保存 checkpoint
        if (epoch + 1) % 20 == 0:
            ts = datetime.now().strftime("%Y%m%d_%H%M%S")
            ckpt_dir = os.path.join(root_ckpt_dir, f"epoch{epoch + 1:03d}_{ts}")
            os.makedirs(ckpt_dir, exist_ok=True)
            torch.save({
                "epoch": epoch + 1,
                "Pc_state": Pc.detach().cpu(),
                "optimizer_state": optimizer.state_dict(),
                "loss": loss.item(),
                "epoch_time": epoch_time,
            }, os.path.join(ckpt_dir, "model.pt"))
            print(f"  [Checkpoint] 模型已保存到: {ckpt_dir}")

    print("[Train] 完成。")

# 2. 新位姿512阵列单探头可微定位

In [ ]:
import os
import time
import math
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# ===============================================================
# 配置
# ===============================================================
DEVICE_ID = 1
device = torch.device(f"cuda:{DEVICE_ID}" if torch.cuda.is_available() else "cpu")
print(f"[INIT] 当前运行设备: {device}")

# 声学参数
vs = 1500.0
delta_time = 120e-9
t_start_idx = 225
t_end_idx   = 415
n_time_sub = t_end_idx - t_start_idx

# --- 优化参数 ---
TOTAL_EPOCHS = 600       
LR_START     = 0.0005
SIGMA_START  = 1.5e-3    
SIGMA_TARGET = 0.1e-3    
TOP_K_CANDIDATES = 15     

# ===============================================================
# 1. 加载数据
# ===============================================================
signal_path = "data/kidney_data_sym_B_512_aligned.txt"
loc_path = "data/kidney_loc_sym_B_512_aligned.txt"
CKPT_PATH = "checkpoints_12/epoch200_20260203_205637/model.pt" 
KEEP_RATIO = 0.002     

# --- A. Phantom 模型 ---
if os.path.exists(CKPT_PATH):
    print(f"[LOAD] Model: {CKPT_PATH}")
    ckpt = torch.load(CKPT_PATH, map_location=device)
    Pc_raw = ckpt["Pc_state"].to(device).contiguous().requires_grad_(False)
    
    GRID_SIZE = 256
    coords = (np.arange(GRID_SIZE) - (GRID_SIZE-1)/2.0) * 0.1e-3 
    xg, yg, zg = np.meshgrid(coords, coords, coords, indexing="ij")
    
    src_x_raw = torch.tensor(xg.ravel(), device=device, dtype=torch.float32).contiguous()
    src_y_raw = torch.tensor(yg.ravel(), device=device, dtype=torch.float32).contiguous()
    src_z_raw = torch.tensor(zg.ravel(), device=device, dtype=torch.float32).contiguous()
    
    n_keep = int(Pc_raw.numel() * KEEP_RATIO)
    val_top, idx_top = torch.topk(torch.abs(Pc_raw), n_keep)
    
    Pc_fixed = Pc_raw[idx_top].contiguous()
    xyz_src = torch.stack([src_x_raw[idx_top], src_y_raw[idx_top], src_z_raw[idx_top]], dim=1).contiguous()
    xyz_src_cpu = xyz_src.cpu().numpy()
else:
    print(f"[WARN] 无模型文件，使用随机数据...")
    xyz_src_cpu = np.random.rand(5000, 3) * 0.05 - 0.025
    xyz_src = torch.tensor(xyz_src_cpu, dtype=torch.float32, device=device)
    Pc_fixed = torch.tensor(np.random.rand(5000), dtype=torch.float32, device=device)

# --- B. 传感器数据 ---

signals_all_raw = np.loadtxt(signal_path)
loc_all_raw = np.loadtxt(loc_path)

loc_all_raw = loc_all_raw * 1.0e-3  
signals_all_raw = signals_all_raw/100

sel_step = 1  # 每隔 sel_step 取一个探头，1 表示全取
sel_idx = np.arange(0, loc_all_raw.shape[0], sel_step)
loc_all_raw = loc_all_raw[sel_idx]  # (N_sensors, 3)
signals_all_raw = signals_all_raw[sel_idx] 

signals_all_tensor = torch.tensor(signals_all_raw, dtype=torch.float32, device=device)
NUM_SENSORS = signals_all_raw.shape[0]
print(f"[LOAD] 成功加载 {NUM_SENSORS} 个探头数据")


# ===============================================================
# 2. 核心函数
# ===============================================================
def acoustic_simulation_torch(sens_pos, src_pos, Pc, t_start, n_time, dt, vs, a):
    diff = sens_pos.unsqueeze(0) - src_pos
    r = torch.norm(diff, dim=1) + 1e-12
    t_global = t_start + torch.arange(n_time, device=device, dtype=torch.float32)
    rt = r.unsqueeze(1) - vs * (t_global * dt).unsqueeze(0)
    exponent = torch.exp( - (rt**2) / (2.0 * a * a) )
    amplitude = Pc.unsqueeze(1) * 0.5 * (rt / r.unsqueeze(1))
    return torch.sum(amplitude * exponent, dim=0)

def negative_correlation_loss(pred, target):
    pred_mean, target_mean = pred - pred.mean(), target - target.mean()
    correlation = torch.sum(pred_mean * target_mean) / (torch.norm(pred_mean) * torch.norm(target_mean) + 1e-9)
    return 1.0 - correlation

def find_top_k_candidates(src_pos, Pc, target_sig, bounds, step, k=5):
    xs = torch.arange(bounds[0][0], bounds[0][1], step)
    ys = torch.arange(bounds[1][0], bounds[1][1], step)
    zs = torch.arange(bounds[2][0], bounds[2][1], step)
    grid_x, grid_y, grid_z = torch.meshgrid(xs, ys, zs, indexing='ij')
    candidates = torch.stack([grid_x.flatten(), grid_y.flatten(), grid_z.flatten()], dim=1).to(device)
    
    losses = []
    with torch.no_grad():
        for i in range(0, len(candidates), 500):
            batch = candidates[i:i+500]
            for pos in batch:
                pred = acoustic_simulation_torch(pos, src_pos, Pc, t_start_idx, n_time_sub, delta_time, vs, a=0.5e-3)
                losses.append(negative_correlation_loss(pred, target_sig).item())
    
    top_indices = torch.topk(torch.tensor(losses), k, largest=False).indices
    return candidates[top_indices].cpu().numpy()

def solve_sensor(true_pos, full_sig):
    target_sig = full_sig[t_start_idx:t_end_idx]
    
    # 1. 粗搜
    starts = find_top_k_candidates(xyz_src, Pc_fixed, target_sig, 
                                   [[-0.1,0.1], [-0.1,0.1], [-0.1,0.1]], 0.02, k=TOP_K_CANDIDATES)
    
    # 2. 细搜 (Top-K 竞争)
    competitors = []
    for start_pos in starts:
        param = torch.nn.Parameter(torch.tensor(start_pos, device=device), requires_grad=True)
        opt = torch.optim.Adam([param], lr=LR_START)
        hist = [start_pos]
        
        for epoch in range(TOTAL_EPOCHS):
            opt.zero_grad()
            sigma = SIGMA_START * (SIGMA_TARGET/SIGMA_START)**min(1.0, epoch/(TOTAL_EPOCHS*0.8))
            pred = acoustic_simulation_torch(param, xyz_src, Pc_fixed, t_start_idx, n_time_sub, delta_time, vs, a=sigma)
            loss = negative_correlation_loss(pred, target_sig)
            loss.backward()
            opt.step()
            hist.append(param.detach().cpu().numpy().copy())
            
        with torch.no_grad():
            final_pred = acoustic_simulation_torch(param, xyz_src, Pc_fixed, t_start_idx, n_time_sub, delta_time, vs, a=SIGMA_TARGET)
            
        competitors.append({'loss': loss.item(), 'pos': hist[-1], 'hist': np.array(hist), 'sig': final_pred.cpu().numpy()})
    
    competitors.sort(key=lambda x: x['loss'])
    best = competitors[0]
    return {'true': true_pos, 'pred': best['pos'], 'hist': best['hist'], 'pred_sig': best['sig'], 'true_sig': target_sig.cpu().numpy(), 'corr': 1-best['loss']}

def set_axes_equal(ax):
    limits = np.array([ax.get_xlim3d(), ax.get_ylim3d(), ax.get_zlim3d()])
    origin = np.mean(limits, axis=1)
    radius = 0.5 * np.max(np.abs(limits[:, 1] - limits[:, 0]))
    ax.set_xlim3d([origin[0] - radius, origin[0] + radius])
    ax.set_ylim3d([origin[1] - radius, origin[1] + radius])
    ax.set_zlim3d([origin[2] - radius, origin[2] + radius])

# ===============================================================
# 3. 主循环 (Notebook 显示模式)
# ===============================================================
all_predictions = []

print("\n" + "="*80)
print("开始处理所有探头 (Results will display sequentially below)")
print("="*80 + "\n")

for i in range(NUM_SENSORS):
    # --- 计算 ---
    res = solve_sensor(loc_all_raw[i], signals_all_tensor[i])
    err_mm = np.linalg.norm(res['pred'] - res['true']) * 1000
    all_predictions.append(res['pred'])
    
    # --- 打印数值结果 ---
    print(f"Sensor ID: {i}")
    print(f"  > True: [{res['true'][0]*1000:.2f}, {res['true'][1]*1000:.2f}, {res['true'][2]*1000:.2f}] mm")
    print(f"  > Pred: [{res['pred'][0]*1000:.2f}, {res['pred'][1]*1000:.2f}, {res['pred'][2]*1000:.2f}] mm")
    print(f"  > Error: {err_mm:.3f} mm")
    print("-" * 50)
    
    # --- 绘图 (左右两张) ---
    fig = plt.figure(figsize=(16, 6))
    
    # 左图: 3D 轨迹
    ax1 = fig.add_subplot(1, 2, 1, projection='3d')
    stride = max(1, len(xyz_src_cpu)//2000)
    ax1.scatter(xyz_src_cpu[::stride,0], xyz_src_cpu[::stride,1], xyz_src_cpu[::stride,2], c='gray', s=1, alpha=0.1) # Phantom
    ax1.plot(res['hist'][:,0], res['hist'][:,1], res['hist'][:,2], c='blue', alpha=0.6, label='Trajectory')
    ax1.scatter(res['true'][0], res['true'][1], res['true'][2], c='green', s=80, marker='o', label='Ground Truth')
    ax1.scatter(res['pred'][0], res['pred'][1], res['pred'][2], c='red', s=80, marker='^', label='Prediction')
    ax1.plot([res['true'][0], res['pred'][0]], [res['true'][1], res['pred'][1]], [res['true'][2], res['pred'][2]], 'k--', alpha=0.5)
    ax1.set_title(f"3D Trajectory (Err: {err_mm:.2f}mm)")
    ax1.legend()
    set_axes_equal(ax1)
    
    # 右图: 信号对比
    ax2 = fig.add_subplot(1, 2, 2)
    t = np.arange(n_time_sub)
    ax2.plot(t, res['true_sig'], 'k', alpha=0.5, linewidth=2, label='Ground Truth')
    ax2.plot(t, res['pred_sig'], 'r--', linewidth=1.5, label='Predicted')
    ax2.set_title(f"Signal Comparison (Corr: {res['corr']:.4f})")
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.show() # 直接在 Notebook 单元格里显示
    print("\n") # 空一行，分隔下一个探头

# ===============================================================
# 4. 保存 TXT
# ===============================================================
save_name = "predicted_locations_B_aligned.txt"
np.savetxt(save_name, np.array(all_predictions), fmt='%.6f', header="Predicted x, y, z (meters)")
print("="*80)
print(f"[DONE] 全部处理完毕！所有预测坐标已保存至: {save_name}")

# 3. RANSAC 阵列位姿估计

In [ ]:
import numpy as np
import time

def rigid_transform_3D(A, B):
    """ 普通 SVD 求解 R, t """
    assert A.shape == B.shape
    centroid_A = np.mean(A, axis=0)
    centroid_B = np.mean(B, axis=0)
    AA = A - centroid_A
    BB = B - centroid_B
    H = np.dot(AA.T, BB)
    U, S, Vt = np.linalg.svd(H)
    R = np.dot(Vt.T, U.T)
    if np.linalg.det(R) < 0:
        Vt[2, :] *= -1
        R = np.dot(Vt.T, U.T)
    t = centroid_B.T - np.dot(R, centroid_A.T)
    return R, t.reshape(3, 1)

def get_distances(points):
    """ 计算3个点两两之间的距离 (边长) """
    # points shape: (3, 3)
    d1 = np.linalg.norm(points[0] - points[1])
    d2 = np.linalg.norm(points[0] - points[2])
    d3 = np.linalg.norm(points[1] - points[2])
    return np.array([d1, d2, d3])

def ransac_rigid_transform_robust(src_points, dst_points, threshold=0.005, iterations=1000000):
    """
    针对低内点率优化的 RANSAC
    threshold: 判断内点的距离阈值 (例如 0.005m)
    iterations: 尝试采样的次数
    """
    n_points = src_points.shape[0]
    best_inliers = []
    best_R = np.eye(3)
    best_t = np.zeros((3, 1))
    
    # 用于几何一致性检查的容差 (比内点阈值稍宽一点，防止噪声干扰)
    dist_tolerance = threshold * 2.0 
    
    print(f"开始鲁棒 RANSAC: 总点数 {n_points}, 目标内点阈值 {threshold}m, 迭代 {iterations} 次")
    start_time = time.time()
    
    # 优化：提前计算所有点的坐标，避免循环中重复索引
    src_indices = np.arange(n_points)
    
    valid_sample_count = 0
    
    for i in range(iterations):
        # 1. 随机选取 3 个点 (刚体变换最少需要3点)
        # 使用3点比4点更容易选到纯净样本 (0.05^3 > 0.05^4)
        idx = np.random.choice(n_points, 3, replace=False)
        
        src_sample = src_points[idx]
        dst_sample = dst_points[idx]
        
        # 2. 【核心步骤】几何一致性预检 (Geometric Consistency Check)
        # 如果选中的3个点在 A 中组成的三角形 和 B 中组成的三角形 边长不一样，
        # 说明这3个点里肯定有漂移点，直接跳过 SVD，极大地节省时间并过滤错误样本。
        d_src = get_distances(src_sample)
        d_dst = get_distances(dst_sample)
        
        # 如果任一边长差异超过容差，跳过
        if np.any(np.abs(d_src - d_dst) > dist_tolerance):
            continue
            
        valid_sample_count += 1
        
        # 3. 只有通过了几何检查，才计算变换
        R, t = rigid_transform_3D(src_sample, dst_sample)
        
        # 4. 快速验证：先只看能否拟合这3个采样点 (避免大矩阵运算)
        # sample_transformed = (np.dot(R, src_sample.T) + t).T
        # sample_err = np.linalg.norm(dst_sample - sample_transformed, axis=1)
        # if np.max(sample_err) > threshold:
        #     continue

        # 5. 全局验证
        # 变换公式: P_new = (R @ P_old.T + t).T
        src_transformed = (np.dot(R, src_points.T) + t).T
        errors = np.linalg.norm(dst_points - src_transformed, axis=1)
        
        current_inliers = np.where(errors < threshold)[0]
        
        # 6. 更新最佳模型
        if len(current_inliers) > len(best_inliers):
            best_inliers = current_inliers
            best_R = R
            best_t = t
            print(f"  Iter {i}: 发现更好模型 -> 内点数: {len(best_inliers)}")
            
            # 如果内点数量接近已知的27个，可以提前终止（可选）
            if len(best_inliers) >= 30: 
                print("  已找到足够多的内点，提前结束搜索。")
                break
    
    print(f"RANSAC 结束: 耗时 {time.time()-start_time:.2f}s, 有效几何采样 {valid_sample_count} 次")
    print(f"最终找到内点数: {len(best_inliers)}")
    
    # 7. Refinement: 使用所有找到的内点重新计算 SVD，得到最高精度
    if len(best_inliers) >= 3:
        final_R, final_t = rigid_transform_3D(src_points[best_inliers], dst_points[best_inliers])
        return final_R, final_t, best_inliers
    else:
        return best_R, best_t, best_inliers

# ================= 主程序 =================
# 1. 加载数据
loc_path = "data/kidney_loc_sym_A_512.txt" # 原始形状
pred_path = "predicted_locations_B_aligned.txt"    # 观测数据

try:
    loc_ref = np.loadtxt(loc_path) * 1e-3 # 别忘了转米
    loc_pred = np.loadtxt(pred_path)      # 假设已经是米 (如果文件是mm, 记得 *1e-3)
except Exception as e:
    print(f"数据加载失败: {e}")
    exit()

# 2. 运行增强版 RANSAC
# threshold=0.005 (5mm) 与你刚才验证的一致
# iterations=50000 足够大，因为加了“几何预检”，跑得很快
R_opt, t_opt, inliers = ransac_rigid_transform_robust(loc_ref, loc_pred, threshold=0.005, iterations=10000)

# 3. 结果验证
loc_corrected = (np.dot(R_opt, loc_ref.T) + t_opt).T
diff_final = np.linalg.norm(loc_pred - loc_corrected, axis=1)

print("\n" + "="*80)
print(f"鲁棒算法结果验证")
print("="*80)

# 打印找回的内点ID，看看是否包含你刚才列表里的 ID (如 2, 14, 26, 34...)
inliers_sorted = np.sort(inliers)
print(f"算法认定为 Inliers 的 ID ({len(inliers)}个):\n{inliers_sorted}")

print("-" * 80)
# 计算这些内点的拟合误差
print(f"内点平均误差: {np.mean(diff_final[inliers]):.6f} m")

# 保存
output_path = "predicted_locations_B_aligned_corrected.txt"
np.savetxt(output_path, loc_corrected, fmt='%.6f')
print(f"矫正结果已保存: {output_path}")

# 4. 基于可微辐射进行整体预测阵列位姿finetune（RANSAC判定的内点参与）

In [ ]:
import os
import time
import math
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# ===============================================================
# 配置与初始化
# ===============================================================
DEVICE_ID = 1
device = torch.device(f"cuda:{DEVICE_ID}" if torch.cuda.is_available() else "cpu")
print(f"[INIT] 当前运行设备: {device}")

# --- 声学参数 ---
vs = 1500.0             # 声速
delta_time = 120e-9      # 采样间隔
t_start_idx = 225      # 截取起始点
t_end_idx   = 415      # 截取结束点
n_time_sub = t_end_idx - t_start_idx

# --- 优化参数 ---
TOTAL_EPOCHS = 100      
LR_ROT       = 1e-4      
LR_TRANS     = 1e-4      
SIGMA_FIXED  = 0.1e-3    

# --- 显存优化参数 ---
# 2080Ti 建议设为 8，兼顾速度和显存
MICRO_BATCH_SIZE = 5     

# --- !!! 排除列表 !!! ---
# 这些探头不参与 Loss 计算（不监督），但会随刚体一起移动并保存
# 索引基于 0-64 (即下采样后的索引)
INCLUDE_IDS = [ 2, 14,  38,  46,  63,  81,  86, 159, 160, 161, 207, 217, 229, 237, 247, 270, 272, 304, 314, 317, 336, 343, 371, 396, 420, 478]
# ===============================================================
# 1. 数据加载
# ===============================================================
signal_path = "data/kidney_data_sym_B_512_aligned.txt"
init_array_path = "predicted_locations_B_aligned_corrected.txt" 
CKPT_PATH = "checkpoints_12/epoch200_20260203_205637/model.pt" 
KEEP_RATIO = 0.002     

# --- A. 加载 Phantom 模型 ---
if os.path.exists(CKPT_PATH):
    print(f"[LOAD] Model: {CKPT_PATH}")
    ckpt = torch.load(CKPT_PATH, map_location=device)
    Pc_raw = ckpt["Pc_state"].to(device).contiguous().requires_grad_(False)
    
    GRID_SIZE = 256
    coords = (np.arange(GRID_SIZE) - (GRID_SIZE-1)/2.0) * 0.1e-3 
    xg, yg, zg = np.meshgrid(coords, coords, coords, indexing="ij")
    
    src_x_raw = torch.tensor(xg.ravel(), device=device, dtype=torch.float32).contiguous()
    src_y_raw = torch.tensor(yg.ravel(), device=device, dtype=torch.float32).contiguous()
    src_z_raw = torch.tensor(zg.ravel(), device=device, dtype=torch.float32).contiguous()
    
    n_keep = int(Pc_raw.numel() * KEEP_RATIO)
    val_top, idx_top = torch.topk(torch.abs(Pc_raw), n_keep)
    
    Pc_fixed = Pc_raw[idx_top].contiguous()
    xyz_src = torch.stack([src_x_raw[idx_top], src_y_raw[idx_top], src_z_raw[idx_top]], dim=1).contiguous()
else:
    raise FileNotFoundError(f"找不到模型文件: {CKPT_PATH}")

# --- B. 加载观测信号 ---
signals_full_raw = np.loadtxt(signal_path)
sel_step = 1
sel_idx = np.arange(0, signals_full_raw.shape[0], sel_step) # 129 -> 65
signals_sub_raw = signals_full_raw[sel_idx] 
# 截取时间窗
signals_target = torch.tensor(signals_sub_raw[:, t_start_idx:t_end_idx], dtype=torch.float32, device=device)
NUM_TOTAL_SENSORS = signals_target.shape[0] # 应该为 65

# --- C. 加载初始位置 ---
init_pos_np = np.loadtxt(init_array_path)
if init_pos_np.shape[0] != NUM_TOTAL_SENSORS:
    raise ValueError(f"坐标数量 ({init_pos_np.shape[0]}) 与 信号数量 ({NUM_TOTAL_SENSORS}) 不匹配！")
init_pos_tensor = torch.tensor(init_pos_np, dtype=torch.float32, device=device)

# --- D. 构建有效训练集索引 ---
# 生成 0 到 64
all_indices = np.arange(NUM_TOTAL_SENSORS)
# 排除指定的 ID
active_indices_np = INCLUDE_IDS
# 转为 Tensor 以便后续索引
active_indices_tensor = torch.tensor(active_indices_np, dtype=torch.long, device=device)

print(f"[CONFIG] 总探头数: {NUM_TOTAL_SENSORS}")
print(f"[CONFIG] 包含探头ID: {INCLUDE_IDS}")
print(f"[CONFIG] 实际参与优化的探头数: {len(active_indices_np)}")

# ===============================================================
# 2. 核心函数与模型
# ===============================================================

def build_rotation_matrix(r):
    cx, sx = torch.cos(r[0]), torch.sin(r[0])
    cy, sy = torch.cos(r[1]), torch.sin(r[1])
    cz, sz = torch.cos(r[2]), torch.sin(r[2])
    Rx = torch.stack([torch.tensor([1.,0,0], device=device), torch.stack([torch.tensor(0., device=device), cx, -sx]), torch.stack([torch.tensor(0., device=device), sx, cx])])
    Ry = torch.stack([torch.stack([cy, torch.tensor(0., device=device), sy]), torch.tensor([0.,1,0], device=device), torch.stack([-sy, torch.tensor(0., device=device), cy])])
    Rz = torch.stack([torch.stack([cz, -sz, torch.tensor(0., device=device)]), torch.stack([sz, cz, torch.tensor(0., device=device)]), torch.tensor([0.,0,1], device=device)])
    return Rz @ Ry @ Rx

def vectorized_acoustic_simulation(sens_pos_batch, src_pos, Pc, t_start, n_time, dt, vs, a):
    # (Batch, N_src, 1) - (1, N_src, 1) -> (Batch, N_src, 3) -> norm -> (Batch, N_src)
    diff = sens_pos_batch.unsqueeze(1) - src_pos.unsqueeze(0) 
    r = torch.norm(diff, dim=2) + 1e-12 
    
    t_global = t_start + torch.arange(n_time, device=device, dtype=torch.float32) 
    
    # (Batch, N_src, T)
    rt = r.unsqueeze(2) - vs * (t_global * dt).reshape(1, 1, -1)
    exponent = torch.exp( - (rt**2) / (2.0 * a * a) )
    amplitude = Pc.reshape(1, -1, 1) * 0.5 * (rt / r.unsqueeze(2))
    
    signals = torch.sum(amplitude * exponent, dim=1)
    return signals

def batch_negative_correlation_loss(pred_batch, target_batch):
    pred_mean = pred_batch - pred_batch.mean(dim=1, keepdim=True)
    target_mean = target_batch - target_batch.mean(dim=1, keepdim=True)
    numerator = torch.sum(pred_mean * target_mean, dim=1)
    d1 = torch.norm(pred_mean, dim=1)
    d2 = torch.norm(target_mean, dim=1)
    correlation = numerator / (d1 * d2 + 1e-9)
    return 1.0 - correlation.mean()

class RigidArrayOptimizer(nn.Module):
    def __init__(self, init_points):
        super().__init__()
        self.centroid = torch.mean(init_points, dim=0, keepdim=True).detach()
        self.points_centered = (init_points - self.centroid).detach()
        
        # 优化参数: 旋转和平移
        self.rot_euler = nn.Parameter(torch.zeros(3, device=device)) 
        self.trans_vec = nn.Parameter(torch.zeros(3, device=device))
        
    def forward(self):
        R = build_rotation_matrix(self.rot_euler)
        # 这一步计算了 ALL 65 points 的位置
        p_final = (self.points_centered @ R.T) + self.centroid + self.trans_vec
        return p_final

# ===============================================================
# 3. 训练循环 (带 Mask)
# ===============================================================
print("\n" + "="*80)
print(f"开始优化 (Valid Sensors Only, Total: {len(active_indices_np)})")
print("="*80)

model = RigidArrayOptimizer(init_pos_tensor).to(device)
optimizer = torch.optim.Adam([
    {'params': model.rot_euler, 'lr': LR_ROT},
    {'params': model.trans_vec, 'lr': LR_TRANS}
])

loss_history = []
start_time = time.time()

for epoch in range(TOTAL_EPOCHS):
    optimizer.zero_grad()
    
    # 1. 获得当前所有512个探头的位置 (但只有 active indices 的位置会对 Loss 产生贡献)
    current_pos_all = model()
    
    total_loss_val = 0.0
    num_active = len(active_indices_np)
    
    # 2. 遍历 Active Indices
    for i in range(0, num_active, MICRO_BATCH_SIZE):
        # 获取当前批次在 active 列表中的索引范围
        start_k = i
        end_k = min(i + MICRO_BATCH_SIZE, num_active)
        
        # 获取真实的探头 ID (例如第0个active可能是ID 0, 第25个active可能是ID 26(跳过了25))
        batch_ids = active_indices_tensor[start_k:end_k]
        
        # 切片：只取参与训练的探头位置和真实信号
        pos_batch = current_pos_all[batch_ids]       
        target_batch = signals_target[batch_ids]     
        
        # 仿真
        pred_batch = vectorized_acoustic_simulation(
            pos_batch, xyz_src, Pc_fixed, 
            t_start_idx, n_time_sub, delta_time, vs, a=SIGMA_FIXED
        )
        
        # 计算 Loss
        loss_fragment = batch_negative_correlation_loss(pred_batch, target_batch)
        
        # 加权 & Backward
        weight = (end_k - start_k) / num_active
        weighted_loss = loss_fragment * weight
        weighted_loss.backward(retain_graph=True)
        
        total_loss_val += weighted_loss.item()
        del pred_batch, loss_fragment, weighted_loss
    
    optimizer.step()
    del current_pos_all
    
    loss_history.append(total_loss_val)
    
    if epoch % 2 == 0:
        r_deg = model.rot_euler.detach().cpu().numpy() * 180 / np.pi
        t_mm = model.trans_vec.detach().cpu().numpy() * 1000
        print(f"Epoch {epoch:04d} | Active_Loss: {total_loss_val:.6f} | "
              f"dRot(deg): {r_deg} | "
              f"dTrans(mm): {t_mm}")

print(f"\n[DONE] 优化耗时: {time.time()-start_time:.1f}s")

# ===============================================================
# 4. 结果保存 (保存所有 512 个探头)
# ===============================================================
final_pos_tensor = model().detach() # 获取最终所有点的坐标
final_pos_np = final_pos_tensor.cpu().numpy()

# 验证一下：即使是不训练的 ID (如 25)，位置也应该发生了变化(跟随刚体)
init_pos_cpu = init_pos_tensor.cpu().numpy()
check_idx = 25
move_dist = np.linalg.norm(final_pos_np[check_idx] - init_pos_cpu[check_idx]) * 1000
print(f"\n[CHECK] 排除探头 #{check_idx} 的跟随移动距离: {move_dist:.4f} mm")

# 保存
save_final_name = "predicted_array_location_B_aligned_corrected_finetuned_masked.txt"
np.savetxt(save_final_name, final_pos_np, fmt='%.8f', header="Finetuned x, y, z (meters) - All 512 Sensors")
print(f"[SAVE] 所有512个探头坐标(含未监督的)已保存至: {save_final_name}")

# 画图 (简单展示Active和Inactive的区别)
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# 所有的初始位置
ax.scatter(init_pos_cpu[:,0], init_pos_cpu[:,1], init_pos_cpu[:,2], c='gray', alpha=0.3, label='Initial')

# 所有的最终位置 (active 用红色，inactive 用蓝色)
# 创建颜色数组
colors = ['red'] * NUM_TOTAL_SENSORS
sizes = [20] * NUM_TOTAL_SENSORS
for idx in INCLUDE_IDS:
    colors[idx] = 'blue' # 排除的显示为蓝色
    sizes[idx] = 50      # 排除的画大一点

ax.scatter(final_pos_np[:,0], final_pos_np[:,1], final_pos_np[:,2], c=colors, s=sizes, marker='^', label='Fine-Tuned')

# 仅创建一个图例
from matplotlib.lines import Line2D
legend_elements = [Line2D([0], [0], marker='o', color='w', markerfacecolor='gray', label='Initial'),
                   Line2D([0], [0], marker='^', color='w', markerfacecolor='red', label='Optimized (Active)'),
                   Line2D([0], [0], marker='^', color='w', markerfacecolor='blue', label='Followed (Included)')]
ax.legend(handles=legend_elements)
ax.set_title("Rigid Body Optimization (Blue sensors included in Loss)")
plt.show()

## 未微调位姿结果与GT对比

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # 必须导入以支持3D绘图


sensor_loc_new_pos = np.loadtxt('data/kidney_loc_sym_B_512_aligned.txt')
sensor_loc_new_pos = sensor_loc_new_pos * 1e-3  # 转换为米
# （可选）下采样探头以降低计算量
sel_step = 1  # 每隔 sel_step 取一个探头，1 表示全取
sel_idx = np.arange(0, sensor_loc_new_pos.shape[0], sel_step)
sensor_loc_new_pos_part = sensor_loc_new_pos[sel_idx]  # (N_sensors, 3)

sensor_loc_new_pos_predict = np.loadtxt('predicted_locations_B_aligned_corrected.txt')
# ----------------------
# 1. 计算误差 (Calculate Error)
# ----------------------

# 计算对应点之间的欧几里得距离 (L2 norm)
# axis=1 表示沿着坐标维度 (x,y,z) 计算范数
errors = np.linalg.norm(sensor_loc_new_pos_part - sensor_loc_new_pos_predict, axis=1)

# 计算平均误差
mean_error = np.mean(errors)
max_error = np.max(errors)
min_error = np.min(errors)

print(f"=== 误差统计 ===")
print(f"平均误差 (Mean Error): {mean_error:.6f}")
print(f"最大误差 (Max Error) : {max_error:.6f}")
print(f"最小误差 (Min Error) : {min_error:.6f}")

# ----------------------
# 2. 可视化 (Visualization)
# ----------------------

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# 绘制真实值 (Ground Truth / Part) - 蓝色圆点
ax.scatter(sensor_loc_new_pos_part[:, 0], 
           sensor_loc_new_pos_part[:, 1], 
           sensor_loc_new_pos_part[:, 2], 
           c='blue', marker='o', s=20, label='Ground Truth (Part)')

# 绘制预测值 (Prediction) - 红色三角
ax.scatter(sensor_loc_new_pos_predict[:, 0], 
           sensor_loc_new_pos_predict[:, 1], 
           sensor_loc_new_pos_predict[:, 2], 
           c='red', marker='^', s=20, label='Prediction')

# (可选) 绘制连接线：将对应的真实点和预测点连起来，直观展示偏差方向
# 如果点很多，可以注释掉这部分代码以保持图面整洁
for i in range(len(sensor_loc_new_pos_part)):
    ax.plot([sensor_loc_new_pos_part[i, 0], sensor_loc_new_pos_predict[i, 0]],
            [sensor_loc_new_pos_part[i, 1], sensor_loc_new_pos_predict[i, 1]],
            [sensor_loc_new_pos_part[i, 2], sensor_loc_new_pos_predict[i, 2]],
            color='gray', linestyle='--', linewidth=0.5, alpha=0.5)

# 设置标签和标题
ax.set_xlabel('X Label')
ax.set_ylabel('Y Label')
ax.set_zlabel('Z Label')
ax.set_title(f'Sensor Array Comparison\nMean Error: {mean_error:.4f}')

# 设置图例
ax.legend()

# 调整视角 (可选)
# ax.view_init(elev=20, azim=45) 

plt.tight_layout()
plt.show()

## 微调位姿结果与GT对比

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # 必须导入以支持3D绘图


sensor_loc_new_pos = np.loadtxt('data/kidney_loc_sym_B_512_aligned.txt')
sensor_loc_new_pos = sensor_loc_new_pos * 1e-3  # 转换为米
# （可选）下采样探头以降低计算量
sel_step = 1  # 每隔 sel_step 取一个探头，1 表示全取
sel_idx = np.arange(0, sensor_loc_new_pos.shape[0], sel_step)
sensor_loc_new_pos_part = sensor_loc_new_pos[sel_idx]  # (N_sensors, 3)

sensor_loc_new_pos_predict = np.loadtxt('predicted_array_location_B_aligned_corrected_finetuned_masked.txt')
# ----------------------
# 1. 计算误差 (Calculate Error)
# ----------------------

# 计算对应点之间的欧几里得距离 (L2 norm)
# axis=1 表示沿着坐标维度 (x,y,z) 计算范数
errors = np.linalg.norm(sensor_loc_new_pos_part - sensor_loc_new_pos_predict, axis=1)

# 计算平均误差
mean_error = np.mean(errors)
max_error = np.max(errors)
min_error = np.min(errors)

print(f"=== 误差统计 ===")
print(f"平均误差 (Mean Error): {mean_error:.6f}")
print(f"最大误差 (Max Error) : {max_error:.6f}")
print(f"最小误差 (Min Error) : {min_error:.6f}")

# ----------------------
# 2. 可视化 (Visualization)
# ----------------------

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# 绘制真实值 (Ground Truth / Part) - 蓝色圆点
ax.scatter(sensor_loc_new_pos_part[:, 0], 
           sensor_loc_new_pos_part[:, 1], 
           sensor_loc_new_pos_part[:, 2], 
           c='blue', marker='o', s=20, label='Ground Truth (Part)')

# 绘制预测值 (Prediction) - 红色三角
ax.scatter(sensor_loc_new_pos_predict[:, 0], 
           sensor_loc_new_pos_predict[:, 1], 
           sensor_loc_new_pos_predict[:, 2], 
           c='red', marker='^', s=20, label='Prediction')

# (可选) 绘制连接线：将对应的真实点和预测点连起来，直观展示偏差方向
# 如果点很多，可以注释掉这部分代码以保持图面整洁
for i in range(len(sensor_loc_new_pos_part)):
    ax.plot([sensor_loc_new_pos_part[i, 0], sensor_loc_new_pos_predict[i, 0]],
            [sensor_loc_new_pos_part[i, 1], sensor_loc_new_pos_predict[i, 1]],
            [sensor_loc_new_pos_part[i, 2], sensor_loc_new_pos_predict[i, 2]],
            color='gray', linestyle='--', linewidth=0.5, alpha=0.5)

# 设置标签和标题
ax.set_xlabel('X Label')
ax.set_ylabel('Y Label')
ax.set_zlabel('Z Label')
ax.set_title(f'Sensor Array Comparison\nMean Error: {mean_error:.6f}')

# 设置图例
ax.legend()

# 调整视角 (可选)
# ax.view_init(elev=20, azim=45) 

plt.tight_layout()
plt.show()

# 5. PA-SFM后，双位姿迭代重建

In [ ]:
import os
import time
from datetime import datetime

import numpy as np
import torch
import torch.nn.functional as F
import triton
import triton.language as tl

# ===============================================================
# CUDA 初始化
# ===============================================================
torch.cuda.init()
DEVICE_ID = 1
torch.cuda.set_device(DEVICE_ID)
device = torch.device(f"cuda:{DEVICE_ID}")
print(f"[INIT] 当前运行设备: {device}")

# ===============================================================
# 声学参数 / 分辨率
# ===============================================================
sigma = 0.1e-3     # m，既作为核参数也作为网格步长（分辨率）
vs = 1500.0        # m/s
delta_time = 120e-9 # s, 采样间隔

# ===============================================================
# 仿真时间范围：起始下标 / 结束下标（不包含结束下标）
# ===============================================================
t_start_idx = 225  # 包含.   如不需要，改为 0
t_end_idx   = 415  # 不包含   如不需要，改为 采样点数
# ===============================================================
# 探头位置 & 参考信号（仅用于尺寸和 target）
# ===============================================================
loc_A = np.loadtxt("data/kidney_loc_sym_A_512.txt")
loc_A = loc_A*1.0e-3
signals_A = np.loadtxt("data/kidney_data_sym_A_512.txt") / 100

loc_B = np.loadtxt("predicted_array_location_B_aligned_corrected_finetuned_masked.txt")
signals_B = np.loadtxt("data/kidney_data_sym_B_512_aligned.txt") / 100

loc_all = np.vstack((loc_A, loc_B))
signals_all = np.vstack((signals_A, signals_B))

n_sensors_full, n_time = signals_all.shape
n_time_sub = t_end_idx - t_start_idx
print(f"[INFO] sensors={n_sensors_full}, full time={n_time}, sim time range=({t_start_idx}, {t_end_idx}), len={n_time_sub}")

# （可选）下采样探头以降低计算量
sel_step = 1  # 每隔 sel_step 取一个探头，1 表示全取
sel_idx = np.arange(0, loc_all.shape[0], sel_step)
loc = loc_all[sel_idx]  # (N_sensors, 3)
signals = signals_all[sel_idx][:, t_start_idx:t_end_idx]  # 只取时间段
n_sensors = loc.shape[0]
print(f"[INFO] 使用探头数={n_sensors} (每 {sel_step} 个取一个)")

sens_x = torch.tensor(loc[:, 0], dtype=torch.float32, device=device).contiguous()
sens_y = torch.tensor(loc[:, 1], dtype=torch.float32, device=device).contiguous()
sens_z = torch.tensor(loc[:, 2], dtype=torch.float32, device=device).contiguous()

# ===============================================================
# 源点：基于分辨率 sigma 的 256^3 网格初始化
# ===============================================================
GRID_SIZE = 256
voxel_size = sigma  # 网格步长 = 分辨率
center = (GRID_SIZE - 1) / 2.0
coords = (np.arange(GRID_SIZE) - center) * voxel_size  # 从 -center*voxel_size 到 +center*voxel_size
xg, yg, zg = np.meshgrid(coords, coords, coords, indexing="ij")
src_x_np = xg.astype(np.float32).ravel()
src_y_np = yg.astype(np.float32).ravel()
src_z_np = zg.astype(np.float32).ravel()
n_sources = src_x_np.size
print(f"[INFO] grid sources = {n_sources} ({GRID_SIZE}^3), step={voxel_size} m")

src_x = torch.tensor(src_x_np, device=device).contiguous()
src_y = torch.tensor(src_y_np, device=device).contiguous()
src_z = torch.tensor(src_z_np, device=device).contiguous()

# 源强度 Pc 作为可训练参数
Pc_param = torch.nn.Parameter(torch.randn(n_sources, device=device))

# ===============================================================
# Triton 前向核：Gaussian
# out[s, t] = sum_k Pc[k] * 0.5 * ((r_k - vs*t)/r_k) * exp(-(r_k - vs*t)^2 / (2*sigma^2))
# ===============================================================
@triton.jit
def forward_kernel(
    Pc_ptr, src_x_ptr, src_y_ptr, src_z_ptr,
    sens_x_ptr, sens_y_ptr, sens_z_ptr,
    out_ptr,
    n_sources, n_time_sub,
    t_start_idx,
    delta_t, vs, sigma,
    stride_out_s, stride_out_t,
    BLOCK_K: tl.constexpr, BLOCK_T: tl.constexpr
):
    pid_t = tl.program_id(0)  # 时间块
    pid_s = tl.program_id(1)  # 探头
    t_offsets = tl.arange(0, BLOCK_T)
    t_idx_rel = pid_t * BLOCK_T + t_offsets
    t_mask = t_idx_rel < n_time_sub
    t_vals = (t_start_idx + t_idx_rel) * delta_t  # [T]

    sx = tl.load(sens_x_ptr + pid_s)
    sy = tl.load(sens_y_ptr + pid_s)
    sz = tl.load(sens_z_ptr + pid_s)

    acc = tl.zeros((BLOCK_T,), dtype=tl.float32)

    for k in range(0, n_sources, BLOCK_K):
        idx_k = k + tl.arange(0, BLOCK_K)
        mask_k = idx_k < n_sources

        px = tl.load(src_x_ptr + idx_k, mask=mask_k, other=0.0)
        py = tl.load(src_y_ptr + idx_k, mask=mask_k, other=0.0)
        pz = tl.load(src_z_ptr + idx_k, mask=mask_k, other=0.0)
        pc = tl.load(Pc_ptr    + idx_k, mask=mask_k, other=0.0)

        dx = sx - px
        dy = sy - py
        dz = sz - pz
        r = tl.sqrt(dx * dx + dy * dy + dz * dz + 1e-12)  # [K]

        r_mat  = r[:, None]      # [K,1]
        pc_mat = pc[:, None]     # [K,1]
        t_mat  = t_vals[None, :] # [1,T]

        rt = r_mat - vs * t_mat
        contrib = pc_mat * 0.5 * (rt / r_mat) * tl.exp(-(rt * rt) / (2.0 * sigma * sigma))
        contrib = tl.where(mask_k[:, None], contrib, 0.0)
        acc += tl.sum(contrib, axis=0)

    out_ptrs = out_ptr + pid_s * stride_out_s + t_idx_rel * stride_out_t
    tl.store(out_ptrs, acc, mask=t_mask)

# ===============================================================
# Triton 反向核：对 Pc 求梯度
# ===============================================================
@triton.jit
def backward_kernel(
    src_x_ptr, src_y_ptr, src_z_ptr,
    sens_x_ptr, sens_y_ptr, sens_z_ptr,
    grad_out_ptr, grad_pc_ptr,
    n_sources, n_sensors, n_time_sub,
    t_start_idx,
    delta_t, vs, sigma,
    stride_go_s, stride_go_t,
    BLOCK_K: tl.constexpr, BLOCK_T: tl.constexpr
):
    pid_t = tl.program_id(0)
    pid_s = tl.program_id(1)
    t_offsets = tl.arange(0, BLOCK_T)
    t_idx_rel = pid_t * BLOCK_T + t_offsets
    t_mask = t_idx_rel < n_time_sub
    t_vals = (t_start_idx + t_idx_rel) * delta_t

    sx = tl.load(sens_x_ptr + pid_s)
    sy = tl.load(sens_y_ptr + pid_s)
    sz = tl.load(sens_z_ptr + pid_s)

    go_ptrs = grad_out_ptr + pid_s * stride_go_s + t_idx_rel * stride_go_t
    go = tl.load(go_ptrs, mask=t_mask, other=0.0)  # [T]

    for k in range(0, n_sources, BLOCK_K):
        idx_k = k + tl.arange(0, BLOCK_K)
        mask_k = idx_k < n_sources

        px = tl.load(src_x_ptr + idx_k, mask=mask_k, other=0.0)
        py = tl.load(src_y_ptr + idx_k, mask=mask_k, other=0.0)
        pz = tl.load(src_z_ptr + idx_k, mask=mask_k, other=0.0)

        dx = sx - px
        dy = sy - py
        dz = sz - pz
        r = tl.sqrt(dx * dx + dy * dy + dz * dz + 1e-12)  # [K]

        r_mat = r[:, None]
        t_mat = t_vals[None, :]

        rt = r_mat - vs * t_mat
        dfdpc = 0.5 * (rt / r_mat) * tl.exp(-(rt * rt) / (2.0 * sigma * sigma))
        dfdpc = tl.where(mask_k[:, None], dfdpc, 0.0)

        grad_k = tl.sum(dfdpc * go[None, :], axis=1)  # [K]
        tl.atomic_add(grad_pc_ptr + idx_k, grad_k, mask=mask_k)

# ===============================================================
# 自定义 Autograd Function
# ===============================================================
class GaussianSimFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx,
                Pc, src_x, src_y, src_z,
                sens_x, sens_y, sens_z,
                t_start_idx, n_time_sub,
                delta_t, vs, sigma):
        out = torch.empty((sens_x.numel(), n_time_sub), device=Pc.device, dtype=torch.float32)

        BLOCK_T = 128
        BLOCK_K = 128
        grid = (triton.cdiv(n_time_sub, BLOCK_T), sens_x.numel())

        forward_kernel[grid](
            Pc, src_x, src_y, src_z,
            sens_x, sens_y, sens_z,
            out,
            Pc.numel(), n_time_sub,
            t_start_idx,
            delta_t, vs, sigma,
            out.stride(0), out.stride(1),
            BLOCK_K=BLOCK_K, BLOCK_T=BLOCK_T,
            num_warps=4, num_stages=2
        )
        ctx.save_for_backward(src_x, src_y, src_z, sens_x, sens_y, sens_z)
        ctx.n_sources = Pc.numel()
        ctx.n_sensors = sens_x.numel()
        ctx.n_time_sub = n_time_sub
        ctx.t_start_idx = t_start_idx
        ctx.delta_t = delta_t
        ctx.vs = vs
        ctx.sigma = sigma
        return out

    @staticmethod
    def backward(ctx, grad_out):
        src_x, src_y, src_z, sens_x, sens_y, sens_z = ctx.saved_tensors
        grad_pc = torch.zeros(ctx.n_sources, device=grad_out.device, dtype=torch.float32)

        BLOCK_T = 128
        BLOCK_K = 128
        grid = (triton.cdiv(ctx.n_time_sub, BLOCK_T), ctx.n_sensors)

        backward_kernel[grid](
            src_x, src_y, src_z,
            sens_x, sens_y, sens_z,
            grad_out.contiguous(), grad_pc,
            ctx.n_sources, ctx.n_sensors, ctx.n_time_sub,
            ctx.t_start_idx,
            ctx.delta_t, ctx.vs, ctx.sigma,
            grad_out.stride(0), grad_out.stride(1),
            BLOCK_K=BLOCK_K, BLOCK_T=BLOCK_T,
            num_warps=4, num_stages=2
        )
        return grad_pc, None, None, None, None, None, None, None, None, None, None, None

def gaussian_sim(Pc, src_x, src_y, src_z, sens_x, sens_y, sens_z,
                 t_start_idx, n_time_sub, delta_t, vs, sigma):
    return GaussianSimFunction.apply(Pc, src_x, src_y, src_z,
                                     sens_x, sens_y, sens_z,
                                     t_start_idx, n_time_sub,
                                     delta_t, vs, sigma)

# ===============================================================
# TGV 正则
# ===============================================================
def forward_diff(tensor, dim):
    diff = torch.roll(tensor, shifts=-1, dims=dim) - tensor
    idx = [slice(None)] * tensor.ndim
    idx[dim] = slice(-1, None)
    diff[tuple(idx)] = 0.0
    return diff

def tgv2_regularization(A_flat, grid_shape, alpha0=2.0, alpha1=1.0, eps=1e-8):
    A = A_flat.view(*grid_shape)
    gx = forward_diff(A, 0)
    gy = forward_diff(A, 1)
    gz = forward_diff(A, 2)
    grad_mag = torch.sqrt(gx * gx + gy * gy + gz * gz + eps)
    first_term = grad_mag.mean()

    gxx = forward_diff(gx, 0)
    gyy = forward_diff(gy, 1)
    gzz = forward_diff(gz, 2)
    gxy = 0.5 * (forward_diff(gx, 1) + forward_diff(gy, 0))
    gxz = 0.5 * (forward_diff(gx, 2) + forward_diff(gz, 0))
    gyz = 0.5 * (forward_diff(gy, 2) + forward_diff(gz, 1))
    second_sq = gxx * gxx + gyy * gyy + gzz * gzz \
                + 2.0 * (gxy * gxy + gxz * gxz + gyz * gyz)
    second_term = torch.sqrt(second_sq + eps).mean()
    return alpha1 * first_term + alpha0 * second_term

GRID_SHAPE = (GRID_SIZE, GRID_SIZE, GRID_SIZE)
lambda_tv = 5.0
tgv_alpha0 = 2.0
tgv_alpha1 = 1.0

# ===============================================================
# 训练循环
# ===============================================================
if __name__ == "__main__":
    print("\n[Train] 开始训练循环 (直角坐标 Gaussian)...")
    target = torch.tensor(signals, dtype=torch.float32, device=device).contiguous()

    optimizer = torch.optim.Adam([Pc_param], lr=0.5)
    max_epochs = 200
    root_ckpt_dir = "checkpoints_13"
    os.makedirs(root_ckpt_dir, exist_ok=True)

    for epoch in range(max_epochs):
        t_start = time.time()

        optimizer.zero_grad()
        Pc = F.softplus(Pc_param)  # 保持非负

        y_pred = gaussian_sim(Pc, src_x, src_y, src_z, sens_x, sens_y, sens_z,
                              t_start_idx, n_time_sub, delta_time, vs, sigma)

        data_loss = torch.mean((y_pred - target) ** 2)
        tv_loss = tgv2_regularization(Pc, GRID_SHAPE, alpha0=tgv_alpha0, alpha1=tgv_alpha1)
        loss = data_loss + lambda_tv * tv_loss

        loss.backward()
        optimizer.step()

        epoch_time = time.time() - t_start
        grad_mean = Pc_param.grad.mean().item() if Pc_param.grad is not None else 0.0
        print(f"[Epoch {epoch+1}/{max_epochs}] "
              f"loss={loss.item():.6e}, data={data_loss.item():.6e}, tv={tv_loss.item():.6e}, "
              f"grad_mean={grad_mean:.4e}, time={epoch_time:.2f}s, "
              f"Pc_min={Pc.min().item():.3e}, Pc_mean={Pc.mean().item():.3e}")

        # 保存 checkpoint
        if (epoch + 1) % 20 == 0:
            ts = datetime.now().strftime("%Y%m%d_%H%M%S")
            ckpt_dir = os.path.join(root_ckpt_dir, f"epoch{epoch + 1:03d}_{ts}")
            os.makedirs(ckpt_dir, exist_ok=True)
            torch.save({
                "epoch": epoch + 1,
                "Pc_state": Pc.detach().cpu(),
                "optimizer_state": optimizer.state_dict(),
                "loss": loss.item(),
                "epoch_time": epoch_time,
            }, os.path.join(ckpt_dir, "model.pt"))
            print(f"  [Checkpoint] 模型已保存到: {ckpt_dir}")

    print("[Train] 完成。")